# 06-workflow.ipynb

In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()
llm = init_chat_model('gpt-4.1-mini')

## Prompt Chaining

- 매우 잘 정리된 업무 순서가 있을 경우 사용
- 이전 노드에서 처리한 내용을 `state`에 담아 다음 노드로 전송

In [6]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
# 그림 보는 용
from IPython.display import Image, display

# Graph State
class State(TypedDict):
    topic: str
    joke: str
    improved_joke: str
    final_joke: str

# Nodes
def generated_joke(state: State):
    msg = llm.invoke(f'주제 {state['topic']}에 관련된 짧은 농답 생성')
    return {'joke': msg.content}

def improve_joke(state: State):
    msg = llm.invoke(f'말장난을 추가해서 아래 농담을 더 재밌게 만들어보자. \n {state['joke']}')
    return {'improved_joke': msg.content}    

def polish_joke(state: State):
    msg = llm.invoke(f'아래 농담을 재미있게 뒤틀어 보자. \n {state['improved_joke']}')
    return {'final_joke': msg.content}

# Router
def check_punchline(state: State):
    # ? 나 !가 없으면 농담을 improve 하고, 있으면 그대로 진행
    if '?' in state['joke'] or '!' in state['joke']:
        return 'Pass'
    else:
        return 'Fail'
    
# 조립
workflow = StateGraph(State)

workflow.add_node(generated_joke)
workflow.add_node(improve_joke)
workflow.add_node(polish_joke)

workflow.add_edge(START, 'generated_joke')

workflow.add_conditional_edges(
    'generated_joke',
    check_punchline,
    {
        'Pass': END,
        'Fail': 'improve_joke'
    }
)

workflow.add_edge('improve_joke', 'polish_joke')
workflow.add_edge('polish_joke', END)

graph = workflow.compile()

graph

graph.invoke({'topic': '랭체인'})


{'topic': '랭체인',
 'joke': '랭체인에 대해 짧고 재치 있는 농담 하나 드릴게요!\n\n"랭체인 쓰면 내 질문도 꼬리 잘 물겠네, 마치 강아지 산책하듯이!" 🐕📚\n\n필요하면 더 만들어 드릴게요!'}

## Parallelization (병렬화)
- 여러 node(llm)이 동시에 작업을 진행
- 뭐가 먼저 끝날지 알 수 없음
- 하위 task를 동시에 진행시켜서 속도를 올린다.
- 같은 task를 여러번 동시에 돌려서 신뢰성 확보

In [ ]:
class State(TypedDict):
    topic : str
    joke: str
    story: str
    poem: str
    output: str

# Nodes
def generate_joke(state: State):
    msg = llm.invoke(f'Write a joke about {state['topic']}')
    return {'joke': msg.content}

def generate_story(state: State):
    msg = llm.invoke(f'Write a story about {state['topic']}')
    return {'story': msg.content}

def generate_poem(state: State):
    msg = llm.invoke(f'Write a poem about {state['topic']}')
    return {'poem': msg.content}

def aggregate(state: State):
    output = f"""농담, 이야기, 시
    Joke: {state['joke']}
    Story: {state['story']}
    Poem: {state['poem']}
    """    
    return {'output': output}


workflow = StateGraph(State)
workflow.add_node(generate_joke)
workflow.add_node(generate_story)
workflow.add_node(generate_poem)
workflow.add_node(aggregate)

workflow.add_edge(START, 'generate_joke')
workflow.add_edge(START, 'generate_story')
workflow.add_edge(START, 'generate_poem')

workflow.add_edge('generate_joke', 'aggregate')
workflow.add_edge('generate_story', 'aggregate')
workflow.add_edge('generate_poem', 'aggregate')

workflow.add_edge('aggregate', END)

graph = workflow.compile()
graph.invoke({'topic': '피자'})

{'topic': '피자',
 'joke': '피자 좋아하는 사람이 제일 무서운 순간은?  \n치즈가 녹아내릴 때! 🍕😄',
 'story': '옛날 옛적, 작은 마을에 피자를 아주 사랑하는 소년 민호가 살고 있었습니다. 민호는 피자를 먹는 것뿐만 아니라 직접 만드는 것도 좋아했어요. 매주 주말이면 할머니와 함께 작은 피자 가게로 가서 반죽을 치고, 소스를 바르고, 치즈와 다양한 토핑을 올리는 일을 도왔습니다.\n\n어느 날, 마을에서 ‘피자 대회’가 열린다는 소식이 전해졌습니다. 대회 우승자에게는 특별한 피자 굽는 오븐과 함께 마을 최고의 피자 장인의 칭호가 주어진다고 했습니다. 민호는 할머니와 함께 참가하기로 결심했어요.\n\n민호는 대회를 위해 자신만의 특별한 피자 레시피를 개발하기 시작했습니다. 토마토 소스에 비밀 재료를 넣고, 신선한 채소와 치즈를 아낌없이 사용했지요. 친구들과 이웃들에게 맛을 보여주며 의견도 받았습니다.\n\n마침내 대회 당일, 마을 광장에는 다양한 사람들이 모여들었습니다. 각 참가자들은 저마다의 멋진 피자를 선보였고, 심사위원들은 하나하나 맛보며 심사를 진행했어요. 민호가 만든 피자는 풍부한 맛과 신선한 재료 덕분에 모두의 입맛을 사로잡았습니다.\n\n결과 발표가 있었고, 민호의 피자가 우승을 차지했습니다! 모두가 환호했고, 민호는 할머니와 함께 기쁘게 안았습니다. 그날부터 민호는 마을에서 ‘피자 소년’으로 불리며, 더 많은 사람들에게 자신의 피자를 즐겁게 나누어 주었습니다.\n\n그리고 민호의 피자 가게는 점점 인기를 얻어, 마을 사람들의 사랑을 받는 특별한 장소가 되었답니다. 피자가 사람들을 행복하게 만드는 마법 같은 이야기는 그렇게 시작되었습니다.',
 'poem': 'Golden crust with edges bright,  \nCheese that melts in warm delight,  \nTomato sauce so rich and sweet,  \n피자’s charm—a perfect treat.  \n\nToppi